In [ ]:
# ==============================
# Hyperparameter Optimization for Sa-only case
#
# Tuning the hyperparameter of the DNN model for output: {Sa}
# ==============================

import h5py
import numpy as np
import torch
import torch.nn as nn
import optuna
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import os
import seaborn as sns

# ==============================
# Save directory
# ==============================
SAVE_DIR = "your_directory"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:, 0:1].astype(np.float32)  # Only Sa (first column)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape (Sa only):", Y.shape)

# ==============================
# Normalize
# ==============================
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

Y_mean = Y.mean(axis=0)
Y_std = Y.std(axis=0) + 1e-8
Y = (Y - Y_mean) / Y_std

np.save(f"{SAVE_DIR}/X_mean.npy", X_mean)
np.save(f"{SAVE_DIR}/X_std.npy", X_std)
np.save(f"{SAVE_DIR}/Y_mean_Sa.npy", Y_mean)
np.save(f"{SAVE_DIR}/Y_std_Sa.npy", Y_std)

# ==============================
# Split dataset
# ==============================
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val))

input_dim = X.shape[1]
output_dim = 1  # Only Sa
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# Model
# ==============================

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = int(width / (2 ** i))
            next_dim = max(next_dim, 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# ==============================
# Objective function
# ==============================

def objective(trial):

    width = trial.suggest_categorical("width", [64, 128, 256, 512])
    depth = trial.suggest_int("depth", 2, 7)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    model = ForwardDNN(input_dim, output_dim, width, depth, dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    best_val = 1e10
    patience = 20
    counter = 0

    for epoch in range(200):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_dataset)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val:
            best_val = val_loss
            counter = 0
            torch.save(model.state_dict(), f"{SAVE_DIR}/best_model_Sa.pth")
        else:
            counter += 1
            if counter >= patience:
                break

    return best_val

# ==============================
# Run Optuna
# ==============================

study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50)

print("\nBEST PARAMETERS")
print(study.best_params)

np.save(f"{SAVE_DIR}/best_params_Sa.npy", study.best_params)

# Save trials table
df = study.trials_dataframe()
df.to_csv(f"{SAVE_DIR}/optuna_trials_Sa.csv", index=False)

# ==============================
# Plot: Convergence
# ==============================

vals = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
plt.figure()
plt.plot(vals, marker='o')
plt.xlabel("Trial")
plt.ylabel("Validation Loss (MSE)")
plt.title("Optuna Convergence ($S_a$)")
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/convergence_Sa.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot: Hyperparameter Importance
# ==============================
importance = optuna.importance.get_param_importances(study)
plt.figure()
plt.bar(importance.keys(), importance.values())
plt.ylabel("Importance")
plt.title("Hyperparameter Importance ($S_a$)")
plt.savefig(f"{SAVE_DIR}/importance_Sa.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot: Hyperparameter Importance Heatmap
# ==============================
importance_df = pd.DataFrame(list(importance.items()), columns=['Parameter', 'Importance'])
importance_df = importance_df.set_index('Parameter').T

plt.figure(figsize=(max(6, len(importance)*0.6), 2))
sns.heatmap(importance_df, annot=True, cmap='viridis', vmin=0, vmax=1, cbar=True, fmt=".2f")
plt.title("Hyperparameter Importance Heatmap ($S_a$ + $S_z$)")
plt.yticks([])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/importance_heatmap_Sa.png", dpi=300)
plt.close()

# ==============================
# Plot: 3D Scatter of Search Space
# ==============================
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df['params_width'], df['params_depth'], df['value'], c=df['value'])
ax.set_xlabel('Width')
ax.set_ylabel('Depth')
ax.set_zlabel('Val Loss')
plt.title("Search Space (Sa + S)")
plt.savefig(f"{SAVE_DIR}/search_space_Sa.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ==============================
# FINAL MODEL TRAINING - Sa only
#
# Training the DNN model for output: {Sa}
# ==============================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SAVE_DIR = "your_directory"

# ------------------------------
# Load best hyperparameters for Sa
# ------------------------------

best_params = np.load(f"{SAVE_DIR}/best_params_Sa.npy", allow_pickle=True).item()

# ------------------------------
# Model
# ------------------------------
model = ForwardDNN(
    input_dim,
    output_dim,  # 1 for Sa
    best_params["width"],
    best_params["depth"],
    best_params["dropout"]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

# ------------------------------
# Loss
# ------------------------------

criterion = nn.MSELoss()  # only Sa, no weighting needed

train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=best_params["batch_size"])

train_losses, val_losses = [], []

# ------------------------------
# Training loop
# ------------------------------
n_epochs = 500
patience = 50
best_val = float("inf")
counter = 0

for epoch in range(n_epochs):

    # ---- Train ----
    model.train()
    t_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        t_loss += loss.item() * xb.size(0)

    t_loss /= len(train_dataset)
    train_losses.append(t_loss)

    # ---- Validation ----
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            v_loss += loss.item() * xb.size(0)

    v_loss /= len(val_dataset)
    val_losses.append(v_loss)

    print(f"Epoch {epoch+1}: Train={t_loss:.5f} Val={v_loss:.5f}")

    # ---- Early stopping ----
    if v_loss < best_val:
        best_val = v_loss
        counter = 0
        torch.save(model.state_dict(), f"{SAVE_DIR}/final_model_Sa.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

# ------------------------------
# Plot Learning Curve
# ------------------------------

plt.figure()
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE (Sa)")
plt.title("Learning Curve ($S_a$)")
plt.legend()
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/learning_curve_Sa.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# INFERENCE PIPELINE - Sa only
#
# Inference of the trained DNN model for output: {Sa}
# ==============================

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os
import h5py
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# ==============================
# PATH
# ==============================

SAVE_DIR = "your_directory"

# ==============================
# Load normalization
# ==============================

X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean_Sa.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std_Sa.npy")

# ==============================
# Load best params
# ==============================

best_params = np.load(f"{SAVE_DIR}/best_params_Sa.npy", allow_pickle=True).item()

# ==============================
# Model definition (same as training!)
# ==============================

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = int(width / (2 ** i))
            next_dim = max(next_dim, 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# ==============================
# Load model
# ==============================

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ForwardDNN(
    input_dim=6,
    output_dim=1,  # Sa only
    width=best_params["width"],
    depth=best_params["depth"],
    dropout=best_params["dropout"]
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model_Sa.pth", map_location=device))
model.eval()

# ==============================
# Reload dataset
# ==============================

file_path = f"{SAVE_DIR}/dataset_large.h5"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:, 0:1].astype(np.float32)  # Sa only

# ==============================
# Normalize
# ==============================

X_norm = (X - X_mean) / X_std

# ==============================
# SAME split as training
# ==============================

X_train, X_val_norm, Y_train, Y_val = train_test_split(
    X_norm, Y, test_size=0.2, random_state=42
)

# ==============================
# Inference
# ==============================

with torch.no_grad():
    preds_norm = model(torch.from_numpy(X_val_norm).float().to(device)).cpu().numpy()

# Denormalize
Y_pred = preds_norm * Y_std + Y_mean
Y_true = Y_val

# Optional: clip negatives
Y_pred = np.clip(Y_pred, 0, None)

# ==============================
# Metrics
# ==============================

r2 = r2_score(Y_true[:, 0], Y_pred[:, 0])
rmse = np.sqrt(mean_squared_error(Y_true[:, 0], Y_pred[:, 0]))
print(f"Sa -> R2: {r2:.4f}, RMSE: {rmse:.4f}")

# ==============================
# Parity Plot - Sa
# ==============================

plt.figure(figsize=(5,5))
plt.scatter(Y_true[:,0], Y_pred[:,0], alpha=0.6, color='blue', s=5)

min_val = min(Y_true[:,0].min(), Y_pred[:,0].min())
max_val = max(Y_true[:,0].max(), Y_pred[:,0].max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

plt.xlabel("Actual $S_a$ ($\mu$m)")
plt.ylabel("Predicted $S_a$ ($\mu$m)")
plt.title("Parity Plot ($S_a$)")
plt.grid(True)
plt.axis('equal')
plt.savefig(f"{SAVE_DIR}/parity_Sa_Only.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Feature Sensitivity (Permutation) - Sa only
# ==============================

feature_names = ["vc","fz","eps_r","eps_a","z","ri"]
baseline_loss = ((Y_true - Y_pred)**2).mean(axis=0)
importance_matrix = np.zeros((1, X_val_norm.shape[1]))

for j in range(X_val_norm.shape[1]):
    X_perm = X_val_norm.copy()
    np.random.shuffle(X_perm[:, j])
    with torch.no_grad():
        pred = model(torch.from_numpy(X_perm).float().to(device)).cpu().numpy()
    pred = pred * Y_std + Y_mean
    pred = np.clip(pred, 0, None)
    loss = ((Y_true - pred)**2).mean(axis=0)
    importance_matrix[0, j] = loss - baseline_loss

# Normalize to [0,1]
importance_matrix_norm = importance_matrix / importance_matrix.max()

# ==============================
# Heatmap
# ==============================

plt.figure(figsize=(8,2))
im = plt.imshow(importance_matrix_norm, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, label="Normalized ΔMSE")

# Write values
for i in range(importance_matrix_norm.shape[0]):
    for j in range(importance_matrix_norm.shape[1]):
        plt.text(j, i, f"{importance_matrix_norm[i,j]:.2f}",
                 ha='center', va='center', color='white', fontsize=9)

plt.xticks(range(len(feature_names)), feature_names, rotation=30)
plt.yticks([0], ["Sa"])

plt.title("Feature Sensitivity Matrix (Normalized) ($S_a$)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_importance_matrix_Sa.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# Hyperparameter Optimization for Sa-Sz case
#
# Tuning the hyperparameter of the DNN model for output: {Sa,Sz}
# ==============================


import h5py
import numpy as np
import torch
import torch.nn as nn
import optuna
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import os


SAVE_DIR = "your_directory"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================
# Load dataset
# ==============================

file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ==============================
# Normalize
# ==============================

X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

Y_mean = Y.mean(axis=0)
Y_std = Y.std(axis=0) + 1e-8
Y = (Y - Y_mean) / Y_std

np.save(f"{SAVE_DIR}/X_mean.npy", X_mean)
np.save(f"{SAVE_DIR}/X_std.npy", X_std)
np.save(f"{SAVE_DIR}/Y_mean.npy", Y_mean)
np.save(f"{SAVE_DIR}/Y_std.npy", Y_std)

# ==============================
# Split
# ==============================

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val))

input_dim = X.shape[1]
output_dim = Y.shape[1]
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# Model
# ==============================

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = int(width / (2 ** i))
            next_dim = max(next_dim, 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        # layers.append(nn.ReLU())  # ensures output >= 0
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# ==============================
# Objective
# ==============================

def objective(trial):

    width = trial.suggest_categorical("width", [64, 128, 256, 512])
    depth = trial.suggest_int("depth", 2, 7)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    model = ForwardDNN(input_dim, output_dim, width, depth, dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    best_val = 1e10
    patience = 20
    counter = 0

    for epoch in range(200):

        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)

        val_loss /= len(val_dataset)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val:
            best_val = val_loss
            counter = 0
            torch.save(model.state_dict(), f"{SAVE_DIR}/best_model_trial.pth")
        else:
            counter += 1
            if counter >= patience:
                break

    return best_val

# ==============================
# Run Optuna
# ==============================

study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50)

print("\nBEST PARAMETERS")
print(study.best_params)

np.save(f"{SAVE_DIR}/best_params.npy", study.best_params)

# Save trials table
df = study.trials_dataframe()
df.to_csv(f"{SAVE_DIR}/optuna_trials.csv", index=False)

# ==============================
# Convergence
# ==============================

vals = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
plt.figure()
plt.plot(vals, marker='o')
plt.xlabel("Trial")
plt.ylabel("Validation Loss (MSE)")
plt.title("Optuna Convergence")
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/convergence.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Importance
# ==============================

importance = optuna.importance.get_param_importances(study)
plt.figure()
plt.bar(importance.keys(), importance.values())
plt.ylabel("Importance")
plt.title("Hyperparameter Importance")
plt.savefig(f"{SAVE_DIR}/importance.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Hyperparameter Importance Heatmap
# ==============================

importance = optuna.importance.get_param_importances(study)

# Convert to DataFrame for easier plotting
importance_df = pd.DataFrame(list(importance.items()), columns=['Parameter', 'Importance'])
importance_df = importance_df.set_index('Parameter').T  # transpose for heatmap format

plt.figure(figsize=(max(6, len(importance)*0.6), 2))  # width scales with number of parameters

# Create heatmap
sns.heatmap(importance_df, annot=True, cmap='viridis', vmin=0, vmax=1, cbar=True, fmt=".2f")

plt.title("Hyperparameter Importance Heatmap")
plt.yticks([])  # optional: remove y-axis tick since we only have one row
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/importance_heatmap.png", dpi=300)
plt.close()

# ==============================
# Plot 3: 3D Scatter
# ==============================

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df['params_width'], df['params_depth'], df['value'], c=df['value'])
ax.set_xlabel('Width')
ax.set_ylabel('Depth')
ax.set_zlabel('Val Loss')
plt.title("Search Space")
plt.savefig(f"{SAVE_DIR}/search_space.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import pandas as pd

SAVE_DIR = "your_directory"

# Load saved Optuna trials
df = pd.read_csv(f"{SAVE_DIR}/optuna_trials.csv")

# ==============================
# 3D Hyperparameter Visualization (Clean 5D)
# ==============================

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

# Axes
x = df['params_width']
y = df['params_depth']
z = df['value']  # validation loss

# Additional dimensions
c = df['params_lr']                         # color → learning rate
s = (df['params_dropout'] * 200) + 20       # size → dropout

# Scatter plot (no momentum)
scatter = ax.scatter(
    x, y, z,
    c=c,
    s=s,
    cmap='viridis',
    alpha=0.7
)

# Labels
ax.set_xlabel('Width')
ax.set_ylabel('Depth')
ax.set_zlabel('Validation Loss')
plt.title("Hyperparameter Search Space (5D Visualization)")

# Colorbar
cbar = plt.colorbar(scatter, pad=0.1)
cbar.set_label('Learning Rate')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/search_space_5D.png", dpi=300, bbox_inches='tight')
plt.close()


# ==============================
# Seaborn Histograms
# ==============================

hyperparams = ['lr','dropout','weight_decay','batch_size','width','depth']
hyperparams = [hp for hp in hyperparams if f'params_{hp}' in df.columns]

n_cols = 3
n_rows = (len(hyperparams) + n_cols - 1) // n_cols

plt.figure(figsize=(5*n_cols, 4*n_rows))

for i, hp in enumerate(hyperparams):
    plt.subplot(n_rows, n_cols, i+1)
    sns.histplot(df[f'params_{hp}'], bins=10, kde=True)
    plt.xlabel(hp)
    plt.ylabel("Trials")
    plt.title(f"{hp} Distribution")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/hyperparams_histograms.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ==============================
# DNN model for Sa-Sz case
#
# Training the DNN model for output: {Sa,Sz}
# ==============================
import numpy as np

SAVE_DIR = "your_directory"

best_params = np.load(f"{SAVE_DIR}/best_params.npy", allow_pickle=True).item()



# ------------------------------
# Model
# ------------------------------
model = ForwardDNN(
    input_dim,
    output_dim,
    best_params["width"],
    best_params["depth"],
    best_params["dropout"]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

# ------------------------------
# Loss: optional weighting to balance Sa vs Sz
# ------------------------------

loss_weights = torch.tensor([1.0, 0.2], device=device)  # scales Sz contribution
criterion = nn.MSELoss(reduction='none')

train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=best_params["batch_size"])

train_losses, val_losses = [], []

# ------------------------------
# Training loop
# ------------------------------

n_epochs = 500
patience = 50
best_val = float("inf")
counter = 0

for epoch in range(n_epochs):

    # ---- Train ----
    model.train()
    t_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)

        # Weighted MSE loss
        loss = criterion(pred, yb)
        loss = (loss * loss_weights).mean()
        loss.backward()
        optimizer.step()

        t_loss += loss.item() * xb.size(0)

    t_loss /= len(train_dataset)
    train_losses.append(t_loss)

    # ---- Validation ----
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss = (loss * loss_weights).mean()
            v_loss += loss.item() * xb.size(0)

    v_loss /= len(val_dataset)
    val_losses.append(v_loss)

    print(f"Epoch {epoch+1}: Train={t_loss:.5f} Val={v_loss:.5f}")

    # ---- Early stopping ----
    if v_loss < best_val:
        best_val = v_loss
        counter = 0
        torch.save(model.state_dict(), f"{SAVE_DIR}/final_model.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

# ------------------------------
# Plot Learning Curve

# ------------------------------
plt.figure()
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Weighted MSE ($S_a$ + $S_z$)")
plt.title("Learning Curve")
plt.legend()
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/learning_curve_final.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# Inference of the DNN model for Sa-Sz case
#
# Inference of the DNN model for output: {Sa,Sz}
# ==============================

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# ==============================
# PATH
# ==============================

SAVE_DIR = "your_directory"

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ------------------------------
# Compute and save normalization (features + targets)
# ------------------------------
# X: shape (N, 6), Y: shape (N, 2) from your dataset
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

Y_mean = Y.mean(axis=0)      # shape = (2,)
Y_std  = Y.std(axis=0) + 1e-8
Y_norm = (Y - Y_mean) / Y_std

# Save normalization for inference
np.save(f"{SAVE_DIR}/X_mean.npy", X_mean)
np.save(f"{SAVE_DIR}/X_std.npy", X_std)
np.save(f"{SAVE_DIR}/Y_mean.npy", Y_mean)
np.save(f"{SAVE_DIR}/Y_std.npy", Y_std)

X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean.npy")  # shape=(2,)
Y_std  = np.load(f"{SAVE_DIR}/Y_std.npy")   # shape=(2,)

# ==============================
# Load best params
# ==============================

best_params = np.load(f"{SAVE_DIR}/best_params.npy", allow_pickle=True).item()

# ==============================
# Model definition
# ==============================

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = max(int(width / (2 ** i)), 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))  # final layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# ==============================
# Load model
# ==============================

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ForwardDNN(
    input_dim=6,
    output_dim=2,
    width=best_params["width"],
    depth=best_params["depth"],
    dropout=best_params["dropout"]
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model.pth", map_location=device))
model.eval()

# ==============================
# Load dataset
# ==============================

file_path = f"{SAVE_DIR}/dataset_large.h5"
with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

# ==============================
# Normalize features
# ==============================

X_norm = (X - X_mean) / X_std

# ==============================
# Split dataset
# ==============================

X_train, X_val_norm, Y_train, Y_val = train_test_split(
    X_norm, Y, test_size=0.2, random_state=42
)

# ==============================
# Inference
# ==============================

with torch.no_grad():
    preds_norm = model(torch.from_numpy(X_val_norm).float().to(device)).cpu().numpy()

# ==============================
# Denormalize each output separately
# ==============================

Y_pred = preds_norm * Y_std + Y_mean

Y_val_norm = (Y_val - Y_mean) / Y_std

# ==============================
# Metrics
# ==============================

for i, name in enumerate(["Sa", "Sz"]):
    # --- Standardized RMSE ---
    rmse_std = np.sqrt(mean_squared_error(Y_val_norm[:, i], preds_norm[:, i]))

    # --- Original RMSE ---
    rmse = np.sqrt(mean_squared_error(Y_val[:, i], Y_pred[:, i]))

    # --- R2 (always original scale) ---
    r2 = r2_score(Y_val[:, i], Y_pred[:, i])

    print(f"{name} -> R2: {r2:.4f}, RMSE: {rmse:.4f}, RMSE (std): {rmse_std:.4f}")

# ==============================
# Parity plots
# ==============================

def parity_plot(y_true, y_pred, name, save_path, xlim=None, ylim=None, color='blue'):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true, y_pred, alpha=0.6, color=color, s=5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    plt.xlabel(rf"Actual ${name}$ ($\mu$m)")
    plt.ylabel(rf"Predicted ${name}$ ($\mu$m)")
    plt.title(f"Parity Plot - ${name}$")
    plt.grid(True)
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
    plt.axis('equal')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

parity_plot(Y_val[:,0], Y_pred[:,0], "S_a", f"{SAVE_DIR}/parity_Sa.png", xlim=(0,8), ylim=(0,8), color='blue')
parity_plot(Y_val[:,1], Y_pred[:,1], "S_z", f"{SAVE_DIR}/parity_Sz.png", xlim=(0,50), ylim=(0,50), color='green')

# ==============================
# Feature Sensitivity (Permutation)
# ==============================

feature_names = ["vc","fz","eps_r","eps_a","z","ri"]
output_names  = ["Sa","Sz"]

baseline_loss = ((Y_val - Y_pred)**2).mean(axis=0)
importance_matrix = np.zeros((2, X_val_norm.shape[1]))

for j in range(X_val_norm.shape[1]):
    X_perm = X_val_norm.copy()
    np.random.shuffle(X_perm[:, j])

    with torch.no_grad():
        pred_norm = model(torch.from_numpy(X_perm).float().to(device)).cpu().numpy()

    # Denormalize
    pred = pred_norm * Y_std + Y_mean
    pred = np.clip(pred, 0, None)

    loss = ((Y_val - pred)**2).mean(axis=0)
    importance_matrix[:, j] = loss - baseline_loss

# Normalize importance to [0,1]
importance_matrix_norm = importance_matrix / importance_matrix.max()

# ==============================
# Heatmap
# ==============================

plt.figure(figsize=(8,3))
im = plt.imshow(importance_matrix_norm, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im, label="Normalized ΔMSE")
for i in range(importance_matrix_norm.shape[0]):
    for j in range(importance_matrix_norm.shape[1]):
        plt.text(j, i, f"{importance_matrix_norm[i,j]:.2f}",
                 ha='center', va='center', color='white', fontsize=9)
plt.xticks(range(len(feature_names)), feature_names, rotation=30)
plt.yticks(range(len(output_names)), output_names)
plt.title("Feature Sensitivity Matrix (Normalized)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_importance_matrix_norm.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# Inverse design with GRID SEARCH for output: {Sa,Sz}
# ==============================

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from itertools import product

# ------------------------------
# SETTINGS
# ------------------------------
SAVE_DIR = "your_directory"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Target outputs
target_Sa = 3.78
target_Sz = 30.077

# Grid settings
N_grid_per_var = 20
top_k = 20

# Tolerances (original scale)
tol_Sa = 0.1
tol_Sz = 1.0

# ------------------------------
# LOAD NORMALIZATION
# ------------------------------
X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std.npy")

# ------------------------------
# LOAD MODEL
# ------------------------------
best_params = np.load(f"{SAVE_DIR}/best_params.npy", allow_pickle=True).item()

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = max(int(width / (2 ** i)), 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

model = ForwardDNN(
    6, 2,
    best_params["width"],
    best_params["depth"],
    best_params["dropout"]
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model.pth", map_location=device))
model.eval()

# ------------------------------
# SEARCH SPACE
# ------------------------------
bounds = {
    "vc":   (100, 300),
    "fz":   (0.1, 0.9),
    "eps_r":(-0.026, 0.011),
    "eps_a":(0.003, 0.009)
}

z_values  = [2.0, 3.0, 4.0]
ri_values = [3.0, 4.0, 5.0]

# Continuous grids
vc_grid    = np.linspace(*bounds["vc"], N_grid_per_var)
fz_grid    = np.linspace(*bounds["fz"], N_grid_per_var)
eps_r_grid = np.linspace(*bounds["eps_r"], N_grid_per_var)
eps_a_grid = np.linspace(*bounds["eps_a"], N_grid_per_var)

# ------------------------------
# CONSISTENT WEIGHTS (FROM TRAINING)
# ------------------------------
w_Sa = 1.0
w_Sz = 0.2

# Normalize tolerances
tol_Sa_norm = tol_Sa / Y_std[0]
tol_Sz_norm = tol_Sz / Y_std[1]

# ------------------------------
# GRID SEARCH
# ------------------------------
solutions = []

for z, ri in product(z_values, ri_values):
    for vc, fz, eps_r, eps_a in product(vc_grid, fz_grid, eps_r_grid, eps_a_grid):

        x = np.array([vc, fz, eps_r, eps_a, z, ri], dtype=np.float32)
        x_norm = (x - X_mean) / X_std

        with torch.no_grad():
            pred = model(torch.from_numpy(x_norm).float().unsqueeze(0).to(device)).cpu().numpy()[0]

        # Denormalize outputs
        y_pred = pred * Y_std + Y_mean
        Sa_pred, Sz_pred = y_pred

        # ------------------------------
        # NORMALIZED + WEIGHTED ERROR
        # ------------------------------
        Sa_err = (Sa_pred - target_Sa) / Y_std[0]
        Sz_err = (Sz_pred - target_Sz) / Y_std[1]

        err = np.sqrt(
            w_Sa * Sa_err**2 +
            w_Sz * Sz_err**2
        )

        # ------------------------------
        # FILTERING (CONSISTENT)
        # ------------------------------
        if abs(Sa_err) < tol_Sa_norm and abs(Sz_err) < tol_Sz_norm:
            solutions.append((x, y_pred, err))

# ------------------------------
# SORT & SELECT TOP SOLUTIONS
# ------------------------------
solutions = sorted(solutions, key=lambda s: s[2])
best_solutions = solutions[:top_k]

print(f"\nFound {len(solutions)} valid solutions")
print(f"Top {len(best_solutions)} shown:\n")

for i, (x, y, err) in enumerate(best_solutions):
    print(f"--- Solution {i+1} ---")
    print(f"Inputs: vc={x[0]:.2f}, fz={x[1]:.3f}, eps_r={x[2]:.4f}, eps_a={x[3]:.4f}, z={x[4]}, ri={x[5]}")
    print(f"Outputs: Sa={y[0]:.3f}, Sz={y[1]:.3f}")
    print(f"Weighted Error: {err:.4f}\n")

# ------------------------------
# VISUALIZATION
# ------------------------------
if len(solutions) > 0:
    Sa_vals = [s[1][0] for s in solutions]
    Sz_vals = [s[1][1] for s in solutions]

    plt.figure(figsize=(6,5))
    plt.scatter(Sa_vals, Sz_vals, alpha=0.6)
    plt.scatter(target_Sa, target_Sz, color='red', s=100, label="Target")
    plt.xlabel("Sa")
    plt.ylabel("Sz")
    plt.title("Inverse Design Solutions (Grid Search - Weighted)")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{SAVE_DIR}/inverse_solutions_grid_weighted.png", dpi=300)
    plt.show()
else:
    print("⚠️ No solutions found — increase N_grid_per_var or relax tolerances")

In [ ]:
# ==============================
# Inverse design with BAYESIAN OPTIMIZATION for output: {Sa,Sz}
# ==============================

import numpy as np
import torch
import torch.nn as nn
import optuna
import matplotlib.pyplot as plt
import pandas as pd

# ------------------------------
# SETTINGS
# ------------------------------
SAVE_DIR = "your_directory"
device = "cuda" if torch.cuda.is_available() else "cpu"

target_Sa = 3.36
target_Sz = 19.6

n_trials = 500
top_k = 20

# ------------------------------
# LOAD NORMALIZATION
# ------------------------------
X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std.npy")

# ------------------------------
# LOAD MODEL
# ------------------------------
best_params = np.load(f"{SAVE_DIR}/best_params.npy", allow_pickle=True).item()

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = max(int(width / (2 ** i)), 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

model = ForwardDNN(
    6, 2,
    best_params["width"],
    best_params["depth"],
    best_params["dropout"]
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model.pth", map_location=device))
model.eval()

# ------------------------------
# CONSISTENT WEIGHTS (TRAINING)
# ------------------------------
w_Sa = 1.0
w_Sz = 0.2

# ------------------------------
# OBJECTIVE FUNCTION
# ------------------------------
def objective(trial):

    # ------------------------------
    # HARD CONSTRAINTS (search space)
    # ------------------------------
    vc    = trial.suggest_float("vc", 100, 300)  # narrower range
    fz    = trial.suggest_float("fz", 0.1, 0.9)
    eps_r = trial.suggest_float("eps_r", -0.026, 0.011)
    eps_a = trial.suggest_float("eps_a", 0.003, 0.009)

    # Discrete constraints
    z  = trial.suggest_categorical("z", [2.0, 3.0, 4.0])   # only 2,3, 4 inserts
    ri = trial.suggest_categorical("ri", [3.0, 4.0, 5.0])  # restrict radii

    # ------------------------------
    # CONDITIONAL CONSTRAINT
    # ------------------------------
    # Example: if z = 2 → vc must be <= 200

    # if z == 2.0 and vc > 200:
    # if z != 2.0 and ri != 5.0:
        # return 1e6  # large penalty (reject region)

    # ------------------------------
    # MODEL PREDICTION
    # ------------------------------
    x = np.array([[vc, fz, eps_r, eps_a, z, ri]], dtype=np.float32)
    x_norm = (x - X_mean) / X_std

    with torch.no_grad():
        pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

    y_pred = pred * Y_std + Y_mean
    Sa_pred, Sz_pred = y_pred

    # ------------------------------
    # MAIN LOSS (CONSISTENT WITH TRAINING)
    # ------------------------------
    Sa_err = (Sa_pred - target_Sa) / Y_std[0]
    Sz_err = (Sz_pred - target_Sz) / Y_std[1]

    loss = (
        w_Sa * Sa_err**2 +
        w_Sz * Sz_err**2
    )

    # ------------------------------
    # OPTIONAL: SOFT CONSTRAINT (penalty)
    # ------------------------------
    # Example: prefer vc near 180

    # vc_target = 180
    # loss += 0.01 * (vc - vc_target)**2

    return loss

# ------------------------------
# RUN OPTUNA
# ------------------------------
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=n_trials)

# ------------------------------
# SAVE TRIALS
# ------------------------------
df = study.trials_dataframe()
df.to_csv(f"{SAVE_DIR}/inverse_optuna_trials.csv", index=False)

# ------------------------------
# BEST SOLUTION
# ------------------------------
print("\n=== BEST SOLUTION ===")
print("Parameters:", study.best_params)
print("Loss:", study.best_value)

# ------------------------------
# PRINT TOP-K SOLUTIONS
# ------------------------------
df_sorted = df.sort_values("value").reset_index(drop=True)

print(f"\n=== TOP {top_k} SOLUTIONS ===\n")

for i in range(top_k):
    row = df_sorted.iloc[i]

    print(f"--- Solution {i+1} ---")
    print(f"Loss: {row['value']:.6f}")

    print("Inputs:")
    print(f"  vc   = {row['params_vc']:.3f}")
    print(f"  fz   = {row['params_fz']:.3f}")
    print(f"  eps_r= {row['params_eps_r']:.5f}")
    print(f"  eps_a= {row['params_eps_a']:.5f}")
    print(f"  z    = {row['params_z']}")
    print(f"  ri   = {row['params_ri']}")

    # Recompute prediction
    x = np.array([[row['params_vc'], row['params_fz'],
                   row['params_eps_r'], row['params_eps_a'],
                   row['params_z'], row['params_ri']]], dtype=np.float32)

    x_norm = (x - X_mean) / X_std

    with torch.no_grad():
        pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

    y_pred = pred * Y_std + Y_mean

    print("Outputs:")
    print(f"  Sa = {y_pred[0]:.4f}")
    print(f"  Sz = {y_pred[1]:.4f}")
    print()


Sa_list = []
Sz_list = []

for t in study.trials:
    if t.value is None:
        continue

    p = t.params
    x = np.array([[p["vc"], p["fz"], p["eps_r"], p["eps_a"], p["z"], p["ri"]]], dtype=np.float32)
    x_norm = (x - X_mean) / X_std

    with torch.no_grad():
        pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

    y = pred * Y_std + Y_mean
    Sa_list.append(y[0])
    Sz_list.append(y[1])

plt.figure(figsize=(6,5))
plt.scatter(Sa_list, Sz_list, alpha=0.6)
plt.scatter(target_Sa, target_Sz, color='red', s=100, label="Target")
plt.xlabel("$S_a$ ($\mu$m)")
plt.ylabel("$S_z$ ($\mu$m)")
plt.title("(DNN) Solution Space")
plt.legend()
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/inverse_solution_space_dnn.png", dpi=300)
plt.show()


In [ ]:
# ==============================
# Inverse design with DIVERSITY + GLOBAL EXPLORATION for output: {Sa,Sz}
# ==============================

import numpy as np
import torch
import torch.nn as nn
import optuna
import matplotlib.pyplot as plt
import pandas as pd

# ------------------------------
# SETTINGS
# ------------------------------
SAVE_DIR = "your_directory"
device = "cuda" if torch.cuda.is_available() else "cpu"

target_Sa = 6
target_Sz = 33

n_trials = 200   # per (z, ri)
top_k = 10        # per (z, ri)

# Diversity strength
lambda_div = 0.1

# ------------------------------
# LOAD NORMALIZATION
# ------------------------------
X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std.npy")

# ------------------------------
# LOAD MODEL
# ------------------------------
best_params = np.load(f"{SAVE_DIR}/best_params.npy", allow_pickle=True).item()

class ForwardDNN(nn.Module):
    def __init__(self, input_dim, output_dim, width, depth, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for i in range(depth):
            next_dim = max(int(width / (2 ** i)), 16)
            layers += [
                nn.Linear(prev, next_dim),
                nn.BatchNorm1d(next_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = next_dim
        layers.append(nn.Linear(prev, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

model = ForwardDNN(
    6, 2,
    best_params["width"],
    best_params["depth"],
    best_params["dropout"]
).to(device)

model.load_state_dict(torch.load(f"{SAVE_DIR}/final_model.pth", map_location=device))
model.eval()

# ------------------------------
# LOSS WEIGHTS
# ------------------------------
w_Sa = 1.0
w_Sz = 0.2
# w_Sz = 0.6

# ------------------------------
# DIVERSITY STORAGE
# ------------------------------
visited_points = []

def diversity_penalty(x):
    if len(visited_points) == 0:
        return 0.0
    dists = [np.linalg.norm(x - vp) for vp in visited_points]
    return np.exp(-min(dists))  # higher if closer

# ------------------------------
# STORAGE FOR ALL RESULTS
# ------------------------------
all_results = []

# ------------------------------
# LOOP OVER DISCRETE VARIABLES
# ------------------------------
for z_fixed in [2.0, 3.0, 4.0]:
    for ri_fixed in [3.0, 4.0, 5.0]:

        print(f"\n=== Optimizing for z={z_fixed}, ri={ri_fixed} ===")

        # Reset diversity per configuration
        visited_points = []

        def objective(trial):

            vc    = trial.suggest_float("vc", 100, 300)
            fz    = trial.suggest_float("fz", 0.1, 0.9)
            eps_r = trial.suggest_float("eps_r", -0.026, 0.011)
            eps_a = trial.suggest_float("eps_a", 0.003, 0.009)

            x = np.array([[vc, fz, eps_r, eps_a, z_fixed, ri_fixed]], dtype=np.float32)
            x_norm = (x - X_mean) / X_std

            with torch.no_grad():
                pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

            y_pred = pred * Y_std + Y_mean
            Sa_pred, Sz_pred = y_pred

            # MAIN LOSS
            Sa_err = (Sa_pred - target_Sa) / Y_std[0]
            Sz_err = (Sz_pred - target_Sz) / Y_std[1]

            loss = w_Sa * Sa_err**2 + w_Sz * Sz_err**2

            # DIVERSITY PENALTY
            x_raw = np.array([vc, fz, eps_r, eps_a, z_fixed, ri_fixed])
            loss += lambda_div * diversity_penalty(x_raw)

            return loss

        # ------------------------------
        # SAMPLER (MORE EXPLORATION)
        # ------------------------------
        sampler = optuna.samplers.TPESampler(
            n_startup_trials=80,
            multivariate=True,
            group=True,
            seed=42
        )

        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(objective, n_trials=n_trials)

        df = study.trials_dataframe()
        df_sorted = df.sort_values("value").reset_index(drop=True)

        # ------------------------------
        # STORE TOP-K PER CONFIG
        # ------------------------------
        for i in range(top_k):
            row = df_sorted.iloc[i]

            x = np.array([[row['params_vc'], row['params_fz'],
                           row['params_eps_r'], row['params_eps_a'],
                           z_fixed, ri_fixed]], dtype=np.float32)

            x_norm = (x - X_mean) / X_std

            with torch.no_grad():
                pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

            y_pred = pred * Y_std + Y_mean

            all_results.append({
                "z": z_fixed,
                "ri": ri_fixed,
                "vc": row['params_vc'],
                "fz": row['params_fz'],
                "eps_r": row['params_eps_r'],
                "eps_a": row['params_eps_a'],
                "Sa": y_pred[0],
                "Sz": y_pred[1],
                "loss": row['value']
            })

# ------------------------------
# FINAL RESULTS
# ------------------------------
df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("loss").reset_index(drop=True)

df_all.to_csv(f"{SAVE_DIR}/inverse_diverse_solutions.csv", index=False)

print("\n=== FINAL DIVERSE SOLUTIONS ===\n")
print(df_all.head(20))

# ==============================
# LANDSCAPE VISUALIZATION (VALLEYS & PEAKS)
# ==============================

def compute_loss(vc, fz, eps_r, eps_a, z, ri):
    x = np.array([[vc, fz, eps_r, eps_a, z, ri]], dtype=np.float32)
    x_norm = (x - X_mean) / X_std

    with torch.no_grad():
        pred = model(torch.from_numpy(x_norm).float().to(device)).cpu().numpy()[0]

    y_pred = pred * Y_std + Y_mean
    Sa_pred, Sz_pred = y_pred

    Sa_err = (Sa_pred - target_Sa) / Y_std[0]
    Sz_err = (Sz_pred - target_Sz) / Y_std[1]

    return w_Sa * Sa_err**2 + w_Sz * Sz_err**2


# ------------------------------
# GRID SETTINGS
# ------------------------------
vc_vals = np.linspace(100, 300, 60)
fz_vals = np.linspace(0.1, 0.9, 60)

VC, FZ = np.meshgrid(vc_vals, fz_vals)

# Fix remaining variables (adjust if needed)
eps_r_fixed = 0.0
eps_a_fixed = 0.006

# ------------------------------
# FINAL RESULTS TABLE FIGURE WITH MANUAL COLUMN WIDTHS
# ------------------------------

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("loss").reset_index(drop=True)

# Top-K solutions
top_table = df_all.head(20).copy()

# Add numbering column
top_table.insert(0, "No.", range(1, len(top_table) + 1))

# Format each column as needed
top_table["vc"]    = top_table["vc"].map("{:.2f}".format)       # integer
top_table["fz"]    = top_table["fz"].map("{:.3f}".format)      # 3 decimals
top_table["eps_r"] = top_table["eps_r"].map("{:.4f}".format)   # 4 decimals
top_table["eps_a"] = top_table["eps_a"].map("{:.4f}".format)   # 4 decimals
top_table["Sa"]    = top_table["Sa"].map("{:.2f}".format)      # 2 decimals
top_table["Sz"]    = top_table["Sz"].map("{:.2f}".format)      # 2 decimals
top_table["loss"]  = top_table["loss"].map("{:.4f}".format)    # 4 decimals
top_table["z"]     = top_table["z"].map("{:.0f}".format)       # integer
top_table["ri"]    = top_table["ri"].map("{:.0f}".format)      # integer

# ------------------------------
# RENAME COLUMNS (SEPARATE STEP)
# ------------------------------
top_table = top_table.rename(columns={
    "vc": r"$v_c$",
    "fz": r"$f_z$",
    "eps_r": r"$\epsilon_r$",
    "eps_a": r"$\epsilon_a$",
    "Sa": r"$S_a$",
    "Sz": r"$S_z$",
    "z": r"$z$",
    "ri": r"$r_i$",
    "loss": "Loss"
})

# ------------------------------
# CREATE FIGURE TABLE
# ------------------------------
fig, ax = plt.subplots(figsize=(14,6))
ax.axis('off')  # hide axes

# Create table
tbl = ax.table(
    cellText=top_table.values,
    colLabels=top_table.columns,
    cellLoc='center',
    loc='center'
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)

# ------------------------------
# MANUALLY SET COLUMN WIDTHS
# ------------------------------
# Widths in fraction of figure width, sum roughly <=1
col_widths = [0.03, 0.03, 0.03, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.06]
# Columns: No., vc, fz, eps_r, eps_a, Sa, Sz, z, ri

for i, width in enumerate(col_widths):
    for key, cell in tbl.get_celld().items():
        if key[1] == i:  # key = (row, col)
            cell.set_width(width)

plt.title("(DNN) Top-20 Inverse Design Solutions", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/top_solutions_table.png", dpi=300)
plt.show()

# ------------------------------
# 1. CONVERGENCE PLOTS
# ------------------------------
values = [t.value for t in study.trials if t.value is not None]

# Raw convergence
plt.figure(figsize=(6,4))
plt.plot(values, alpha=0.7)
plt.xlabel("Trial")
plt.ylabel("Loss")
plt.title("Case 2: (DNN) Inverse Design Convergence")
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/convergence.png", dpi=300)
plt.show()



# ------------------------------
# PLOT DIVERSE SOLUTIONS WITH COLORBAR
# ------------------------------
plt.figure(figsize=(6,5))

# Assign a unique id per (z, ri) combination
df_all["config_id"] = df_all.groupby(["z", "ri"]).ngroup()

scatter = plt.scatter(
    df_all["Sa"],
    df_all["Sz"],
    c=df_all["config_id"],
    cmap="tab10",
    alpha=0.8
)

# Target point in black
plt.scatter(target_Sa, target_Sz, color='black', s=100, label="Target")

# Colorbar for configuration
cbar = plt.colorbar(scatter)
cbar.set_label("Configuration (z, $r_i$)")

# Custom ticks showing (z, ri)
configs = df_all[["config_id", "z", "ri"]].drop_duplicates().sort_values("config_id")
cbar.set_ticks(configs["config_id"])
cbar.set_ticklabels([f"z={int(z)}, $r_i$={int(ri)}" for z, ri in zip(configs["z"], configs["ri"])])

plt.xlabel("$S_a$ ($\\mu$m)")
plt.ylabel("$S_z$ ($\\mu$m)")
plt.title("(DNN) Inverse Design Solution Space")
plt.legend()
plt.grid(True)

plt.savefig(f"{SAVE_DIR}/inverse_diverse_solution_space.png", dpi=300)
plt.show()